# OMGM Step 3: Section-Augmented Knowledge-Based VQA Answer Generation
### ACL 2025 Main (Paper Baseline Replication) - 100% Reproducible

This notebook performs **Step 3** of the OMGM pipeline using the **exact paper architecture**:
1. **Stage 3A (Section Reranking)**: Uses `BAAI/bge-reranker-v2-m3` blended with Step 2 fine-grained section scores (beta = 0.2) to select the single most relevant Wikipedia section.
2. **Stage 3B (Answer Generation)**: Uses `llava-hf/llava-1.5-7b-hf` (greedy deterministic decoding, seed=42) to answer the visual question conditioned on the selected section.
3. **Evaluation & Taxonomy Breakdown**: Evaluates exact match / numerical range accuracy and categorizes all failures into **Category A, B, C, and D**.

### Cell 1: Environment & Dependencies
Installs required HuggingFace generation packages and clones the OMGM repository.

In [ ]:
# 1. Install required lightweight packages
!pip install -q transformers accelerate sentencepiece

import os, sys
REPO_DIR = '/kaggle/working/OMGM'
if not os.path.exists(REPO_DIR):
    print('Cloning OMGM repository...')
    !git clone --depth 1 https://github.com/ChaoLinAViy/OMGM /kaggle/working/OMGM

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

### Cell 2: Deterministic Seeds & Path Configuration
Sets random seeds (42) for 100% deterministic reproducibility and verifies dataset paths.

In [ ]:
import os, sys, csv, json, time, random, glob, re, string
from typing import Dict, List, Optional
import torch
import numpy as np
import PIL.Image
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoProcessor, LlavaForConditionalGeneration

# Set Seed for 100% Deterministic Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Active Device: {device}')

# Dataset Paths
BASE_DS = '/kaggle/input/datasets/alphacentauriv/agentic-vqa-ds'
SAMPLE_FILE = os.path.join(BASE_DS, 'infoseek_test_sample_1000.csv')

# Step 2 Results (res_step2.json)
STEP2_CANDIDATES = [
    os.path.join(BASE_DS, 'res_step2.json'),
    '/kaggle/working/step2_results/result.json',
    os.path.join(BASE_DS, 'result.json')
]
STEP2_RESULT_FILE = None
for p in STEP2_CANDIDATES:
    if os.path.exists(p):
        STEP2_RESULT_FILE = p
        break

KB_PATH = os.path.join(BASE_DS, 'infoseek_100k_wiki/wiki_100_dict_v4.json')
QUERY_IMG_DIR = os.path.join(BASE_DS, 'infoseek_val_1000_images/infoseek_val_1000_images')
OUTPUT_DIR = '/kaggle/working/step3_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Paths verified:')
print('  Sample CSV    :', SAMPLE_FILE, '--> Exists:', os.path.exists(SAMPLE_FILE))
print('  Step 2 Result :', STEP2_RESULT_FILE, '--> Exists:', os.path.exists(STEP2_RESULT_FILE) if STEP2_RESULT_FILE else False)
print('  Wikipedia KB  :', KB_PATH, '--> Exists:', os.path.exists(KB_PATH))
print('  Query Images  :', QUERY_IMG_DIR, '--> Exists:', os.path.exists(QUERY_IMG_DIR))
print('  Output Dir    :', OUTPUT_DIR)

### Cell 3: Helper Functions & Evaluation Metrics
Official InfoSeek evaluation metrics (normalization, string exact match, numerical tolerance) and section parsing.

In [ ]:
class KnowledgeEntry:
    def __init__(self, title, url, section_titles, section_texts):
        self.title = title
        self.url = url
        self.section_titles = section_titles
        self.section_texts = section_texts

def reconstruct_wiki_sections(entry: KnowledgeEntry) -> List[str]:
    sections = []
    title = entry.title
    for it, sec_title in enumerate(entry.section_titles):
        sec_lower = sec_title.lower()
        if 'external links' in sec_lower or 'references' in sec_lower:
            continue
        text = entry.section_texts[it] if it < len(entry.section_texts) else ''
        sections.append(f'# Wiki Article: {title}\n## Section Title: {sec_title}\n{text}')
    return sections

def resolve_query_image(image_id: str, query_img_dir: str) -> str:
    clean_id = image_id.split('|')[0].strip()
    for ext in ['', '.jpg', '.jpeg', '.JPEG', '.png']:
        candidate = os.path.join(query_img_dir, clean_id + ext)
        if os.path.exists(candidate):
            return candidate
    matches = glob.glob(os.path.join(query_img_dir, f'*{clean_id}*'))
    if matches:
        return matches[0]
    raise FileNotFoundError(f'Query image {image_id} not found in {query_img_dir}')

def normalize_answer(text: str) -> str:
    def remove_articles(t: str) -> str:
        return re.sub(r'\b(a|an|the)\b', ' ', t)
    def white_space_fix(t: str) -> str:
        return ' '.join(t.split())
    def remove_punctuation(t: str) -> str:
        return ''.join(ch for ch in t if ch not in set(string.punctuation))
    return white_space_fix(remove_articles(remove_punctuation(text.lower())))

def exact_match_score(pred: str, gt: str) -> bool:
    return normalize_answer(pred) == normalize_answer(gt)

def extract_number(text: str) -> Optional[float]:
    matches = re.findall(r'[-+]?(?:\d*\.\d+|\d+)', text)
    if matches:
        try:
            return float(matches[0])
        except ValueError:
            return None
    return None

def metric_numerical_range(pred_num: float, ans_num: float, tolerance: float = 0.1) -> int:
    min_val = ans_num * (1.0 - tolerance) if ans_num >= 0 else ans_num * (1.0 + tolerance)
    max_val = ans_num * (1.0 + tolerance) if ans_num >= 0 else ans_num * (1.0 - tolerance)
    low, high = min(min_val, max_val), max(min_val, max_val)
    return 1 if low <= pred_num <= high else 0

def evaluate_prediction(pred_str: str, gt_answers: List[str]) -> int:
    if not pred_str:
        return 0
    norm_pred = normalize_answer(pred_str)
    for gt in gt_answers:
        if exact_match_score(pred_str, gt):
            return 1
        norm_gt = normalize_answer(gt)
        if norm_gt and (norm_gt == norm_pred or f' {norm_gt} ' in f' {norm_pred} '):
            return 1
    pred_val = extract_number(pred_str)
    if pred_val is not None:
        for gt in gt_answers:
            gt_val = extract_number(gt)
            if gt_val is not None and metric_numerical_range(pred_val, gt_val):
                return 1
    return 0

# OMGM Exact Few-Shot Prompt Template
VLM_INFOSEEK_INSTRUCTION = (
    'Answer the encyclopedic question about the given image.'
    'Don\'t mention the visuall content of image in your output. '
    'Directly output the answer of the question according to the Context.\n'
    'If you need to answer questions about numbers or time, please output the corresponding numerical format directly.\n'
    'If the context does not contain the information required to answer the question, you should answer the question using internal model knowledge.\n '
    'There is an example:\n- Context: # Wiki Article: Dolomites\n## Section Title: Dolomites\nThe Dolomites, also known as the Dolomite Mountains, Dolomite Alps or Dolomitic Alps, are a mountain range located in northeastern Italy. The Dolomites are located in the regions of Veneto, Trentino-Alto Adige/Südtirol and Friuli Venezia Giulia, covering an area shared between the provinces of Belluno, Vicenza, Verona, Trentino, South Tyrol, Udine and Pordenone.\n'
    '- Question: Which city or region does this mountain locate in?\n'
    'Just answer the questions , no explanations needed. Short answer is: Province of Belluno\n\n'
)

def build_omgm_prompt(question: str, section_text: str, max_chars: int = 3000) -> str:
    trimmed = section_text[:max_chars] if len(section_text) > max_chars else section_text
    return VLM_INFOSEEK_INSTRUCTION + f'- Context: {trimmed}\n- Question: {question}\nJust answer the questions , no explanations needed. Short answer is:'

### Cell 4: Stage 3A - Section Reranking with BGE & Step 2 Blending
Loads `BAAI/bge-reranker-v2-m3` to score sections of the top-1 entity, blends with Step 2's fine-grained section scores (beta = 0.2), picks the best section, and immediately frees BGE memory to prevent GPU OOM.

In [ ]:
# 1. Load Step 2 Results
print('Loading Step 2 results...')
with open(STEP2_RESULT_FILE, 'r', encoding='utf-8') as f:
    step2_results = json.load(f)

# 2. Load Test Samples
with open(SAMPLE_FILE, 'r', encoding='utf-8') as f:
    samples = list(csv.DictReader(f))
print(f'Loaded {len(samples)} test samples.')

# 3. Load Wikipedia KB
print('Loading Wikipedia Knowledge Base...')
t0 = time.time()
with open(KB_PATH, 'r', encoding='utf-8') as f:
    raw_kb = json.load(f)
if isinstance(raw_kb, list):
    kb_dict = {item['url']: KnowledgeEntry(item.get('title', ''), item.get('url', ''), item.get('section_titles', []), item.get('section_texts', [])) for item in raw_kb}
else:
    kb_dict = {url: KnowledgeEntry(info.get('title', ''), url, info.get('section_titles', []), info.get('section_texts', [])) for url, info in raw_kb.items()}
del raw_kb
print(f'Knowledge Base loaded ({len(kb_dict)} articles in {time.time() - t0:.1f}s).\n')

# 4. Load BGE Section Reranker
print('Loading BAAI/bge-reranker-v2-m3 onto CUDA...')
bge_tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-v2-m3')
bge_model = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-v2-m3').to(device).eval()

STEP2_BETA = 0.2
query_selected_sections = {}
t_sec = time.time()

print('Reranking candidate sections across 1,000 queries...')
with torch.no_grad():
    for it, sample in enumerate(tqdm(samples, desc='Stage 3A Section Reranking')):
        data_id = sample.get('data_id', f'query_{it}')
        question = sample.get('question', '')

        step2_item = step2_results.get(data_id) or step2_results.get(str(data_id)) or list(step2_results.values())[it]
        top1_entity_url = step2_item['retrieved_entities'][0]['url']

        if top1_entity_url in kb_dict:
            sections = reconstruct_wiki_sections(kb_dict[top1_entity_url])
        else:
            sections = []

        if len(sections) == 0:
            query_selected_sections[data_id] = {
                'top1_section': 'No relevant section available.',
                'top1_entity_url': top1_entity_url,
                'section_count': 0
            }
            continue

        # Score sections with BGE
        pairs = [[question, sec] for sec in sections]
        q2s_scores = []
        for sp in range(0, len(pairs), 4):
            inp = bge_tokenizer(pairs[sp : sp + 4], padding=True, truncation=True, return_tensors='pt', max_length=2048).to(device)
            logits = bge_model(**inp, return_dict=True).logits.view(-1).float()
            q2s_scores.extend(logits.cpu().tolist())

        # Blend with Step 2 multimodal section scores
        step2_sec_sim = step2_item.get('sec_sim', [[]])[0]
        if len(step2_sec_sim) == len(q2s_scores):
            step3_scores = [STEP2_BETA * float(s2) + (1.0 - STEP2_BETA) * float(bg) for s2, bg in zip(step2_sec_sim, q2s_scores)]
        else:
            step3_scores = q2s_scores

        best_idx = int(np.argmax(step3_scores))
        query_selected_sections[data_id] = {
            'top1_section': sections[best_idx],
            'top1_entity_url': top1_entity_url,
            'section_count': len(sections),
            'chosen_section_index': best_idx
        }

print(f'Stage 3A Section Reranking completed in {time.time() - t_sec:.1f}s.')

# Free BGE Section Reranker from GPU
del bge_model, bge_tokenizer
torch.cuda.empty_cache()
print('Freed BGE Section Reranker from GPU memory. VRAM clean for LLaVA.')

# Save intermediate selected sections
sec_out = os.path.join(OUTPUT_DIR, 'step3_selected_sections.json')
with open(sec_out, 'w', encoding='utf-8') as f:
    json.dump(query_selected_sections, f, indent=2)
print(f'Saved selected sections to: {sec_out}')

### Cell 5: Stage 3B - LLaVA-1.5-7B Answer Generation & Failure Taxonomy
Loads `llava-hf/llava-1.5-7b-hf`, generates concise answers, evaluates official InfoSeek metrics, and performs the complete 4-way failure breakdown (Categories A, B, C, D).

In [ ]:
print('Loading LLaVA-1.5-7B (llava-hf/llava-1.5-7b-hf) onto GPU...')
MODEL_ID = 'llava-hf/llava-1.5-7b-hf'
processor = AutoProcessor.from_pretrained(MODEL_ID)

try:
    generator = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True
    ).to(device).eval()
    print('Successfully loaded LLaVA-1.5-7B in FP16.')
except torch.cuda.OutOfMemoryError:
    print('[Notice] FP16 exceeded GPU VRAM. Falling back to 4-bit quantized LLaVA...')
    generator = LlavaForConditionalGeneration.from_pretrained(
        MODEL_ID, load_in_4bit=True, low_cpu_mem_usage=True
    ).eval()

correct_vqa = 0
cat_a_count = 0
cat_b_count = 0
cat_c_count = 0
cat_d_count = 0
results_summary = []

t_gen = time.time()
print('Starting LLaVA-1.5-7B VQA generation loop across 1,000 queries...')

with torch.no_grad():
    for it, sample in enumerate(tqdm(samples, desc='Stage 3B Answer Generation')):
        data_id = sample.get('data_id', f'query_{it}')
        question = sample.get('question', '')
        ground_truth_url = sample.get('wikipedia_url', '').strip()
        raw_gt_answer = sample.get('answer', '')
        gt_answer_list = [a.strip() for a in raw_gt_answer.split('|') if a.strip()]

        # Load Query Image
        img_id = sample.get('dataset_image_ids', '').split('|')[0]
        try:
            q_img_path = resolve_query_image(img_id, QUERY_IMG_DIR)
            raw_image = PIL.Image.open(q_img_path).convert('RGB')
        except Exception:
            raw_image = PIL.Image.new('RGB', (224, 224), color=(128, 128, 128))

        # Selected Section Context
        sec_info = query_selected_sections[data_id]
        top1_section = sec_info['top1_section']
        top1_entity_url = sec_info['top1_entity_url']

        # Step 2 Entity Status
        step2_item = step2_results.get(data_id) or step2_results.get(str(data_id)) or list(step2_results.values())[it]
        step2_top20_urls = [e['url'] for e in step2_item['retrieved_entities'][:20]]
        entity_in_top20 = ground_truth_url in step2_top20_urls
        entity_is_top1 = (ground_truth_url == top1_entity_url)

        # Build Chat Conversation
        prompt_text = build_omgm_prompt(question, top1_section)
        conversation = [
            {
                'role': 'user',
                'content': [
                    {'type': 'text', 'text': prompt_text},
                    {'type': 'image'},
                ],
            }
        ]
        input_prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
        inputs = processor(images=raw_image, text=input_prompt, return_tensors='pt').to(device)

        # Greedy Deterministic Generation
        output_tokens = generator.generate(**inputs, max_new_tokens=25, do_sample=False)
        raw_decoded = processor.decode(output_tokens[0], skip_special_tokens=True)
        pred_answer = raw_decoded.split('ASSISTANT:')[-1].strip()

        # Evaluate
        is_correct = evaluate_prediction(pred_answer, gt_answer_list)
        if is_correct:
            correct_vqa += 1

        # Classify Failure Mode
        failure_category = 'None (Correct)'
        if not is_correct:
            if not entity_in_top20:
                failure_category = 'Category A (Entity Missing from Top-20)'
                cat_a_count += 1
            elif not entity_is_top1:
                failure_category = 'Category B (Entity in Top-20, but Rank > 1)'
                cat_b_count += 1
            else:
                section_has_evidence = any(
                    normalize_answer(ans) in normalize_answer(top1_section)
                    for ans in gt_answer_list
                )
                if not section_has_evidence:
                    failure_category = 'Category C (Wrong Section Selected)'
                    cat_c_count += 1
                else:
                    failure_category = 'Category D (Generator Reasoning Failure)'
                    cat_d_count += 1

        results_summary.append({
            'data_id': data_id,
            'question': question,
            'prediction': pred_answer,
            'ground_truth_answers': gt_answer_list,
            'is_correct': bool(is_correct),
            'top1_entity_url': top1_entity_url,
            'ground_truth_url': ground_truth_url,
            'failure_category': failure_category
        })

# Save Final VQA Results
vqa_out = os.path.join(OUTPUT_DIR, 'step3_final_vqa_results.json')
with open(vqa_out, 'w', encoding='utf-8') as f:
    json.dump(results_summary, f, indent=2)

total_q = len(samples)
vqa_acc = (correct_vqa / total_q) * 100.0

print('\n' + '=' * 65)
print('FINAL STEP 3 EVALUATION & FAILURE TAXONOMY BREAKDOWN')
print(f'Total Generation Time: {time.time() - t_gen:.1f}s')
print('=' * 65)
print(f'Overall VQA Accuracy: {vqa_acc:.2f}% ({correct_vqa}/{total_q})')
print('-' * 65)
print(f'  Correct Answers               : {correct_vqa:4d} ({correct_vqa/total_q*100:5.2f}%)')
print(f'  Category A (Retrieval Ceiling): {cat_a_count:4d} ({cat_a_count/total_q*100:5.2f}%)')
print(f'  Category B (Reranking Error)  : {cat_b_count:4d} ({cat_b_count/total_q*100:5.2f}%)')
print(f'  Category C (Wrong Section)    : {cat_c_count:4d} ({cat_c_count/total_q*100:5.2f}%)')
print(f'  Category D (Generator Failure): {cat_d_count:4d} ({cat_d_count/total_q*100:5.2f}%)')
print('=' * 65)
print(f'Successfully saved final VQA results to: {vqa_out}')